# Predição de contratação de depósito a prazo — Bank Marketing

**Disciplina:** CMPINAM — Introdução ao Aprendizado de Máquina  
**Tarefa:** classificação binária da variável `subscribed` (`yes`/`no`).  
**Base:** Bank Marketing, disponibilizada para a atividade; origem original: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/222/bank+marketing).  

Este notebook percorre preparação, análise exploratória, validação de modelos e avaliação final. A divisão de teste fornecida pelo docente é preservada e só é usada após a escolha do modelo.

## 1. Base e preparação

Cada linha representa um contato de uma campanha de marketing telefônico de uma instituição bancária portuguesa. Os atributos incluem características demográficas, informações do contato/campanha e indicadores econômicos. `subscribed` informa se houve contratação do depósito a prazo.

A base fornecida tem 19 preditores e uma coluna-alvo. `duration` não está disponível, pois só seria conhecido após a ligação e causaria vazamento. A documentação fornecida informa que `pdays = 999` representa ausência de contato anterior. Essa situação será codificada em um indicador e o sentinela será tratado como ausente para a imputação numérica.

As categorias `unknown` são mantidas como categoria própria: indicam informação não registrada e não equivalem necessariamente a um valor ausente. Os tipos inferidos na leitura (inteiros e números decimais para atributos numéricos; texto para categorias e alvo) foram conferidos; não foi necessária conversão manual. A criação de ausências em `pdays` após tratar o sentinela é intencional. Valores ausentes serão imputados dentro dos pipelines, ajustados apenas com os dados de treino. A remoção de duplicatas exatas ocorre antes da divisão interna para impedir que registros repetidos apareçam nas duas partições.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
DATA_DIR = Path("CMPINAM_Dataset_01_Bank_Marketing")
TRAIN_PATH = DATA_DIR / "bank_marketing_train.csv"
TEST_PATH = DATA_DIR / "bank_marketing_test.csv"
TARGET = "subscribed"
EXPECTED_TARGETS = {"no", "yes"}

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        "Não encontrei os CSVs. Execute o notebook a partir da pasta raiz do projeto."
    )

train_raw = pd.read_csv(TRAIN_PATH)
expected_columns = set(train_raw.columns)
if TARGET not in expected_columns:
    raise ValueError(f"A coluna-alvo obrigatória {TARGET!r} não existe.")

missing_targets = set(train_raw[TARGET].dropna().unique()) - EXPECTED_TARGETS
if missing_targets or train_raw[TARGET].isna().any():
    raise ValueError(f"Valores inesperados na variável-alvo: {missing_targets}")

duplicates_removed = int(train_raw.duplicated().sum())
modeling_train_raw = train_raw.drop_duplicates().copy()
explicit_missing_before = int(modeling_train_raw.isna().sum().sum())

from sklearn.model_selection import train_test_split

X_full_raw = modeling_train_raw.drop(columns=TARGET)
y_full = modeling_train_raw[TARGET]
X_train_raw, X_valid_raw, y_train, y_valid = train_test_split(
    X_full_raw,
    y_full,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_full,
)
train = X_train_raw.assign(**{TARGET: y_train}).copy()
eda_data = train.assign(contratou=train[TARGET].eq("yes").astype("int8"))

# O valor 999 é um sentinela documentado, não uma quantidade real de dias.
train["never_contacted_before"] = train["pdays"].eq(999).astype("int8")
train["pdays"] = train["pdays"].replace(999, np.nan)

invalid_counts = {
    "age fora de 17–98": int((~modeling_train_raw["age"].between(17, 98)).sum()),
    "campaign não positiva": int((modeling_train_raw["campaign"] <= 0).sum()),
    "previous negativo": int((modeling_train_raw["previous"] < 0).sum()),
}
if any(invalid_counts.values()):
    raise ValueError(f"Foram encontrados valores inválidos: {invalid_counts}")

print(f"Linhas originais de treino: {len(train_raw):,}")
print(f"Duplicatas exatas removidas do treino: {duplicates_removed:,}")
print(f"Linhas de treino após limpeza: {len(modeling_train_raw):,}")
print(f"Treino interno: {len(X_train_raw):,} | validação: {len(X_valid_raw):,}")
print(f"Valores ausentes explícitos na origem: {explicit_missing_before:,}")
print(f"Valores ausentes após codificar pdays=999 como NaN: {int(train.isna().sum().sum()):,}")
print(f"Validação de valores fora do domínio documentado: {invalid_counts}")
display(train.dtypes.rename("tipo").to_frame())
display(train[TARGET].value_counts().rename_axis(TARGET).to_frame("quantidade"))

## 2. Análise exploratória

A divisão estratificada treino/validação é feita antes de qualquer transformação, e a análise exploratória usa somente a partição de treino interno. Gráficos de barras facilitam comparar categorias e taxas; histogramas mostram distribuições contínuas; gráficos de linha destacam tendências ordenadas; e o mapa de calor resume associações lineares entre variáveis numéricas. Cada família é usada por tornar a comparação indicada mais legível.

As taxas de contratação são descritivas, não evidência de causalidade. Categorias com poucos registros podem ter taxas instáveis; as contagens aparecem junto às proporções sempre que pertinente.

### Pergunta 1 — Como as classes estão distribuídas?

Um gráfico de barras é adequado para comparar diretamente a frequência das duas classes e evidenciar o desbalanceamento.

In [ ]:
class_counts = train[TARGET].value_counts().reindex(["no", "yes"])
class_rates = train[TARGET].value_counts(normalize=True).reindex(["no", "yes"])
display(pd.DataFrame({"quantidade": class_counts, "proporção": class_rates}))
ax = class_counts.plot(kind="bar", color=["#5975a4", "#55a868"], rot=0)
ax.set(title="Distribuição da variável-alvo", xlabel="Contratou depósito?", ylabel="Contatos")
plt.tight_layout()
plt.show()

### Pergunta 2 — Qual é a distribuição de idade dos clientes?

O histograma mostra o formato e a dispersão da variável numérica idade sem ocultar a concentração de observações.

In [ ]:
display(train["age"].describe().rename("idade").to_frame())
plt.figure(figsize=(8, 4))
plt.hist(train["age"], bins=25, color="#4c72b0", edgecolor="white")
plt.title("Distribuição de idade")
plt.xlabel("Idade (anos)")
plt.ylabel("Contatos")
plt.tight_layout()
plt.show()

### Pergunta 3 — A taxa de contratação varia entre faixas etárias?

Barras de taxa permitem comparar a proporção de `yes` entre grupos de idade, em vez de confundir conversão com o tamanho de cada grupo.

In [ ]:
age_bins = [15, 25, 35, 45, 55, 65, 100]
age_labels = ["16–25", "26–35", "36–45", "46–55", "56–65", "66+"]
age_groups = pd.cut(train["age"], bins=age_bins, labels=age_labels, include_lowest=True)
age_conversion = eda_data.assign(faixa_etaria=age_groups).groupby(
    "faixa_etaria", observed=False
)["contratou"].agg(taxa="mean", contatos="size")
age_conversion["taxa"] *= 100
display(age_conversion.round(2))
ax = age_conversion["taxa"].plot(kind="bar", color="#55a868", rot=0)
ax.set(title="Taxa de contratação por faixa etária", xlabel="Faixa etária", ylabel="Contratação (%)")
plt.tight_layout()
plt.show()

### Pergunta 4 — Quais ocupações apresentam maiores taxas de contratação?

Barras horizontais ordenadas tornam comparáveis as taxas por ocupação, mesmo com nomes de categorias longos; a tabela inclui o volume de contatos.

In [ ]:
job_conversion = eda_data.groupby("job", dropna=False)["contratou"].agg(taxa="mean", contatos="size")
job_conversion["taxa"] *= 100
job_conversion = job_conversion.sort_values("taxa")
display(job_conversion.sort_values("taxa", ascending=False).round(2))
ax = job_conversion["taxa"].plot(kind="barh", figsize=(8, 5), color="#4c72b0")
ax.set(title="Taxa de contratação por ocupação", xlabel="Contratação (%)", ylabel="Ocupação")
plt.tight_layout()
plt.show()

### Pergunta 5 — A escolaridade está associada à contratação?

Barras de taxa são apropriadas para comparar categorias de escolaridade; `unknown` é mantido e explicitado como falta de informação categórica.

In [ ]:
education_conversion = eda_data.groupby("education", dropna=False)["contratou"].agg(taxa="mean", contatos="size")
education_conversion["taxa"] *= 100
display(education_conversion.sort_values("taxa", ascending=False).round(2))
ax = education_conversion["taxa"].sort_values().plot(kind="barh", color="#c44e52")
ax.set(title="Taxa de contratação por escolaridade", xlabel="Contratação (%)", ylabel="Escolaridade")
plt.tight_layout()
plt.show()

### Pergunta 6 — O canal de contato apresenta diferença na taxa observada?

Barras com a taxa de contratação permitem comparar os canais; as contagens ajudam a contextualizar grupos de tamanhos diferentes.

In [ ]:
contact_conversion = eda_data.groupby("contact", dropna=False)["contratou"].agg(taxa="mean", contatos="size")
contact_conversion["taxa"] *= 100
display(contact_conversion.sort_values("taxa", ascending=False).round(2))
ax = contact_conversion["taxa"].sort_values().plot(kind="barh", color="#8172b2")
ax.set(title="Taxa de contratação por canal de contato", xlabel="Contratação (%)", ylabel="Canal")
plt.tight_layout()
plt.show()

### Pergunta 7 — Como a taxa observada varia entre os meses?

O gráfico de linha acompanha a variação mês a mês; a ordem do calendário é definida explicitamente para não ordenar os meses alfabeticamente.

In [ ]:
month_order = ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
month_conversion = eda_data.groupby("month")["contratou"].agg(taxa="mean", contatos="size")
month_conversion = month_conversion.reindex([m for m in month_order if m in month_conversion.index])
month_conversion["taxa"] *= 100
display(month_conversion.round(2))
ax = month_conversion["taxa"].plot(kind="line", marker="o", color="#dd8452")
ax.set(title="Taxa de contratação por mês da campanha", xlabel="Mês", ylabel="Contratação (%)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Pergunta 8 — O número de contatos na campanha atual se relaciona com a contratação?

A linha da taxa média resume a associação ao longo da contagem de contatos; a tabela mostra os tamanhos dos grupos, importantes para interpretar valores extremos.

In [ ]:
campaign_conversion = eda_data.groupby("campaign")["contratou"].agg(taxa="mean", contatos="size")
campaign_conversion["taxa"] *= 100
display(campaign_conversion.head(15).round(2))
plot_data = campaign_conversion.head(15)
ax = plot_data["taxa"].plot(kind="line", marker="o", color="#937860")
ax.set(title="Taxa de contratação por contatos na campanha (até 15)", xlabel="Número de contatos", ylabel="Contratação (%)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Pergunta 9 — O resultado da campanha anterior está associado à contratação atual?

Barras de taxa com contagens comparam os resultados anteriores sem confundir a taxa com o volume de cada categoria.

In [ ]:
previous_conversion = eda_data.groupby("poutcome", dropna=False)["contratou"].agg(taxa="mean", contatos="size")
previous_conversion["taxa"] *= 100
display(previous_conversion.sort_values("taxa", ascending=False).round(2))
ax = previous_conversion["taxa"].sort_values().plot(kind="barh", color="#64b5cd")
ax.set(title="Taxa de contratação por resultado anterior", xlabel="Contratação (%)", ylabel="Resultado anterior")
plt.tight_layout()
plt.show()

### Pergunta 10 — Quais variáveis numéricas apresentam maior associação linear entre si?

O mapa de calor resume correlações de Pearson entre variáveis numéricas. Correlação não implica causalidade e pode não capturar relações não lineares ou entre categorias.

In [ ]:
numeric_for_corr = train.select_dtypes(include="number").copy()
numeric_for_corr[TARGET + "_yes"] = train[TARGET].eq("yes").astype(int)
correlations = numeric_for_corr.corr()
fig, ax = plt.subplots(figsize=(11, 8))
image = ax.imshow(correlations, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(correlations.columns)), correlations.columns, rotation=90)
ax.set_yticks(range(len(correlations.index)), correlations.index)
ax.set_title("Correlação de Pearson — variáveis numéricas")
fig.colorbar(image, ax=ax, label="Correlação")
plt.tight_layout()
plt.show()
target_correlations = correlations[TARGET + "_yes"].drop(TARGET + "_yes").sort_values(key=abs, ascending=False)
display(target_correlations.rename("correlação com subscribed=yes").to_frame().round(3))

### Síntese da análise exploratória

A base é desbalanceada, portanto a acurácia isolada seria enganosa. As taxas por grupos descrevem associações e precisam ser lidas junto às contagens: segmentos pequenos podem produzir estimativas instáveis. O mapa de calor cobre apenas correlações lineares numéricas; não avalia associações entre categorias nem causalidade.

Na partição de treino interno, `yes` representa 11,7% dos contatos. A contratação observada foi maior após um resultado anterior de sucesso (65,9%; 862 contatos) e entre clientes com 66 anos ou mais (45,2%; 374 contatos), mas esses grupos são menores e essas relações não são causais. O canal celular teve taxa de 15,4% (15.689 contatos), ante 5,5% no telefone (9.527 contatos). As variações mensais também são grandes e algumas se baseiam em poucas observações, então devem ser interpretadas com cautela.

Limitações: dados históricos de Portugal (2008–2010), possíveis mudanças no comportamento e no contexto econômico, campos `unknown`, observações repetidas de contatos/campanhas e ausência de variáveis que poderiam explicar a decisão. A remoção de duplicatas exatas no treino evita que registros idênticos apareçam em treino e validação; não removemos valores extremos plausíveis, pois idade alta e muitas tentativas podem ser observações reais e não há justificativa para descartá-las.

## 3. Modelagem e validação

A tarefa é prever `subscribed`. A divisão do treino fornecido em treino interno e validação estratificados é feita antes de transformar atributos; a EDA usa somente o treino interno. O teste oficial não participa da exploração, seleção de modelo ou ajuste de transformações.

As imputações, a padronização numérica e o one-hot encoding ficam em `Pipeline`/`ColumnTransformer`, portanto são ajustados somente em cada partição de treino. Categorias inéditas são ignoradas pelo encoder. Não foi feita seleção de atributos: mantivemos os preditores disponíveis para evitar descartar informação sem evidência prévia. O desbalanceamento é tratado com pesos de classe nos modelos treinados, sem reamostrar os dados. Não há busca de hiperparâmetros: os parâmetros são fixados para manter o experimento reprodutível e comparável. A validação usa average precision (área sob a curva precisão-revocação) como critério principal, pois a classe positiva é minoritária; também reportamos ROC-AUC, precisão, revocação e F1 no limiar padrão de 0,5.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def engineer_features(features):
    transformed = features.copy()
    transformed["never_contacted_before"] = transformed["pdays"].eq(999).astype("int8")
    transformed["pdays"] = transformed["pdays"].replace(999, np.nan)
    return transformed

X_train = engineer_features(X_train_raw)
X_valid = engineer_features(X_valid_raw)
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = [column for column in X_train.columns if column not in numeric_features]

numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

model_specs = {
    "Baseline (classe majoritária)": DummyClassifier(strategy="most_frequent"),
    "Regressão logística": LogisticRegression(
        class_weight="balanced", max_iter=1000, solver="liblinear", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

def score_model(model, features, labels):
    predictions = model.predict(features)
    positive_index = list(model.classes_).index("yes")
    probabilities = model.predict_proba(features)[:, positive_index]
    return {
        "PR-AUC (average precision)": average_precision_score(
            labels, probabilities, pos_label="yes"
        ),
        "ROC-AUC": roc_auc_score(labels, probabilities),
        "Precisão": precision_score(labels, predictions, pos_label="yes", zero_division=0),
        "Revocação": recall_score(labels, predictions, pos_label="yes", zero_division=0),
        "F1": f1_score(labels, predictions, pos_label="yes", zero_division=0),
    }

validation_results = {}
fitted_models = {}
for model_name, estimator in model_specs.items():
    pipeline = Pipeline(
        steps=[("preprocess", preprocessor), ("model", estimator)]
    )
    pipeline.fit(X_train, y_train)
    fitted_models[model_name] = pipeline
    validation_results[model_name] = score_model(pipeline, X_valid, y_valid)

validation_table = pd.DataFrame(validation_results).T.sort_values(
    ["PR-AUC (average precision)", "F1"], ascending=False
)
display(validation_table.round(3))
best_model_name = validation_table.index[0]
print(f"Modelo escolhido somente pela validação: {best_model_name}")

## 4. Avaliação final no teste fornecido

Depois de selecionar o modelo pela average precision na validação interna, ajustamos novamente o pipeline escolhido usando todo o treino limpo. Só agora carregamos e avaliamos o conjunto de teste fornecido, sem reamostragem ou ajuste de limiar. O teste é mantido integralmente, inclusive eventuais linhas duplicadas, para respeitar a partição oficial.

In [ ]:
test = pd.read_csv(TEST_PATH)
if set(test.columns) != expected_columns:
    raise ValueError("As colunas do teste não correspondem às colunas do treino.")
if test[TARGET].isna().any() or not set(test[TARGET].unique()).issubset(EXPECTED_TARGETS):
    raise ValueError("A variável-alvo do teste contém valores ausentes ou inesperados.")

X_test = test.drop(columns=TARGET)
y_test = test[TARGET]
X_test = engineer_features(X_test)
X = engineer_features(X_full_raw)
y = y_full
if list(X_test.columns) != list(X.columns):
    raise ValueError("A ordem ou os nomes das variáveis preditoras mudaram.")

final_model = fitted_models[best_model_name]
final_model.fit(X, y)
test_metrics = score_model(final_model, X_test, y_test)
print(f"Modelo avaliado no teste: {best_model_name}")
print(f"Registros do teste preservados: {len(test):,}")
display(pd.DataFrame([test_metrics], index=[best_model_name]).round(3))

test_predictions = final_model.predict(X_test)
matrix = confusion_matrix(y_test, test_predictions, labels=["no", "yes"])
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(matrix, cmap="Blues")
ax.set_xticks([0, 1], ["no", "yes"])
ax.set_yticks([0, 1], ["no", "yes"])
ax.set_xlabel("Classe prevista")
ax.set_ylabel("Classe real")
ax.set_title("Matriz de confusão — teste final")
for row in range(2):
    for column in range(2):
        ax.text(column, row, f"{matrix[row, column]:,}", ha="center", va="center")
fig.colorbar(image, ax=ax, label="Contatos")
plt.tight_layout()
plt.show()
print(classification_report(y_test, test_predictions, labels=["no", "yes"], zero_division=0))

## 5. Interpretação, limitações e melhorias

A taxa de acerto global não deve ser interpretada sozinha: a classe `no` é majoritária e um classificador trivial já pode parecer bom nesse indicador. A average precision resume o compromisso entre precisão e revocação para a classe `yes`; a matriz de confusão explicita quantos interessados foram identificados e quantos contatos seriam classificados incorretamente.

O modelo foi escolhido na validação e testado uma única vez. Os resultados não provam que uma característica cause contratação. Melhorias possíveis incluem validação cruzada estratificada no treino, busca de hiperparâmetros com orçamento definido, avaliação de limiares conforme o custo de falso positivo/falso negativo, calibração das probabilidades e validação temporal para investigar mudança de distribuição. A base não contém `duration`; essa omissão reduz o risco de vazamento, mas também limita a informação disponível.